<a href="https://colab.research.google.com/github/Amazeble/chatterbox-finetuning/blob/main/chatterbox-finetuning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Chatterbox TTS Fine-Tuning on Google Colab

This notebook provides a complete workflow for fine-tuning Chatterbox TTS models (Standard and Turbo) with LoRA support.

## Overview
- **Step 1**: Clone repository
- **Step 2**: Connect Google Drive and Copy Dataset
- **Step 3**: Install Dependencies
- **Step 4**: Configure Training Parameters (Select LoRA/Full Fine-tune, Turbo/Non-Turbo)
- **Step 5**: Download & Prepare Models (setup.py)
- **Step 6**: Preprocess Dataset
- **Step 7**: Train the Model
- **Step 8**: Inference (Generate Speech)
- **Step 9**: Merge LoRA Adapter (Optional)

### ⚠️ Important Notes
- **GPU Runtime Required**: Go to `Runtime > Change runtime type > GPU`
- **LoRA Recommended**: For datasets < 10 hours, use LoRA (`is_lora = True`)
- **Turbo Mode**: Faster training but may require LoRA for stability


## Step 1: Clone Repository

In [ ]:
#@title 📥 Clone Chatterbox Repository
#@markdown This cell clones the Chatterbox TTS repository from GitHub.

import os

# Detect environment and set working directory
if os.path.exists('/content'):
    # Running in Google Colab
    WORKSPACE = '/content'
    %cd /content
    print("Detected Google Colab environment")
else:
    # Running locally or in other environments
    WORKSPACE = '/workspace'
    %cd "$WORKSPACE"
    print("Running in local/other environment")

# Clone the repository if it doesn't exist
if not os.path.exists('chatterbox-finetuning'):
    print("Cloning Chatterbox Finetuning repository...")
    !git clone https://github.com/Amazeble/chatterbox-finetuning.git
    %cd chatterbox-finetuning
    print("✅ Repository cloned successfully!")
else:
    print("✅ Repository already exists.")
    %cd chatterbox-finetuning

# Verify contents
print("\nRepository contents:")
!ls -la

## Step 2: Connect Google Drive and Copy Dataset

In [ ]:
#@title 📁 Connect Google Drive and Copy Dataset
#@markdown This cell mounts Google Drive and copies your dataset to the working directory.

import os
from google.colab import drive

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define source and destination paths
source_path = '/content/drive/MyDrive/MyTTSDataset/'
dest_path = '/content/chatterbox-finetuning/'

# Check if source path exists
if os.path.exists(source_path):
    print(f"\n✅ Source path found: {source_path}")
    print("Copying dataset to working directory...")
    
    # Create destination directory if it doesn't exist
    os.makedirs(dest_path, exist_ok=True)
    
    # Copy files
    !cp -r {source_path}/* {dest_path}
    
    print(f"✅ Dataset copied successfully to {dest_path}")
    print("\nVerifying copied files:")
    !ls -la {dest_path}
else:
    print(f"⚠️ Source path not found: {source_path}")
    print("Please ensure your dataset is in the correct location in Google Drive.")

## Step 3: Install Dependencies

In [ ]:
#@title 🔧 Install Dependencies
#@markdown This cell installs all required Python packages and FFmpeg.

import os

# Install FFmpeg
print("Installing FFmpeg...")
!apt-get update && apt-get install -y ffmpeg

# Install Python dependencies
print("\nInstalling Python dependencies...")
%cd /content/chatterbox-finetuning
!pip install -r requirements.txt

print("\n✅ Dependencies installed successfully!")

## Step 4: Configure Training Parameters

In [ ]:
#@title ⚙️ Configure Training Parameters
#@markdown Edit the src/config.py file to set your training preferences.
#@markdown **NOTE**: You will need to run Step 5 (setup.py) after this to download models!
#@markdown After running setup.py, if using Turbo mode, come back here to update vocab_size.

import os

# Check if repository exists
repo_path = '/content/chatterbox-finetuning'
if not os.path.exists(repo_path):
    raise FileNotFoundError(f"❌ Repository not found at {repo_path}! Please run Step 1 (Clone Repository) first.")

%cd /content/chatterbox-finetuning

# Interactive Configuration Widgets
from IPython.display import display, Markdown
import ipywidgets as widgets

print("="*50)
print("🎛️ Training Configuration")
print("="*50)

# Create widgets for key settings
is_turbo_widget = widgets.Checkbox(
    value=True,
    description='Turbo Mode',
    disabled=False,
    tooltip='Enable Turbo mode for faster training'
)

is_lora_widget = widgets.Checkbox(
    value=True,
    description='LoRA Training',
    disabled=False,
    tooltip='Use LoRA (recommended for <10h data). Uncheck for Full Fine-tune'
)

display(Markdown("**Select Training Options:**"))
display(is_turbo_widget)
display(is_lora_widget)

# Display current config
print("
" + "="*50)
print("Current configuration (src/config.py):")
print("="*50)
!cat src/config.py

print("
" + "="*50)
print("To modify settings using the widgets above, run this cell again with your selections.")
print("Key settings:")
print("  - is_turbo: True/False (Turbo mode)")
print("  - is_lora: True/False (LoRA training - recommended for <10h data)")
print("  - new_vocab_size: Must match tokenizer.json (update after setup.py if Turbo)")
print("  - ljspeech/json_format: Dataset format")
print("  - preprocess: Set to True for first run")

# Apply widget settings to config.py
turbo_value = "True" if is_turbo_widget.value else "False"
lora_value = "True" if is_lora_widget.value else "False"

!sed -i "s/is_turbo: bool = .*/is_turbo: bool = /" src/config.py
!sed -i "s/is_lora: bool = .*/is_lora: bool = /" src/config.py

print(f"
✅ Applied settings: is_turbo={turbo_value}, is_lora={lora_value}")
print("⚠️ IMPORTANT: Now proceed to Step 5 to run setup.py!")
print("After setup.py completes, if using Turbo mode, check the output for vocab_size.")
print("You may need to update new_vocab_size in src/config.py manually.")

## Step 5: Download & Prepare Models (setup.py)

In [ ]:
#@title 📥 Download & Prepare Models (setup.py)
#@markdown This cell runs setup.py to download the base models and prepare the environment.

%cd /content/chatterbox-finetuning

print("Running setup.py to download models...")
print("This may take several minutes depending on your internet connection.")

# Run setup.py
!python setup.py

print("\n✅ Setup completed!")
print("Check the output above for the vocab_size if using Turbo mode.")
print("You may need to update new_vocab_size in src/config.py with the value from setup.py output.")

## Step 6: Preprocess Dataset

In [ ]:
#@title 🔄 Preprocess Dataset
#@markdown This cell preprocesses your dataset for training. Make sure preprocess=True in config.py first!

%cd /content/chatterbox-finetuning

print("Preprocessing dataset...")
print("This may take several minutes depending on your dataset size.")

# Run preprocessing by executing train.py with preprocess=True
# The preprocessing happens automatically when preprocess=True in config.py
!python train.py

print("\n✅ Preprocessing completed!")
print("If you want to train without preprocessing next time, set preprocess=False in config.py")

## Step 7: Train the Model

In [ ]:
#@title 🏋️ Train the Model
#@markdown Start the fine-tuning process. This will take several hours depending on your dataset and GPU.

%cd /content/chatterbox-finetuning

print("Starting training...")
print("Training progress and audio samples will be displayed below.")
print("Press Ctrl+C to stop training early.")

# Start training
!python train.py

print("\n✅ Training completed!")
print("Check the chatterbox_output/ directory for your trained model.")

## Step 8: Inference (Generate Speech)

In [ ]:
#@title 🗣️ Inference - Generate Speech
#@markdown Test your trained model by generating speech from text.

%cd /content/chatterbox-finetuning

# First, let's check what models are available
print("Checking available models...")
!ls -la chatterbox_output/

print("\n" + "="*50)
print("To generate speech, edit inference.py with your desired text:")
print("  TEXT_TO_SAY = \"Your text here\"")
print("  AUDIO_PROMPT = \"./speaker_reference/reference.wav\"")
print("="*50)

# Run inference
print("\nRunning inference...")
!python inference.py

print("\n✅ Speech generated!")
print("Output saved as output.wav")

# Play the generated audio
from IPython.display import Audio, display
print("\nPlaying generated audio:")
display(Audio('./output.wav'))

## Step 9: Merge LoRA Adapter (Optional)

In [ ]:
#@title 📦 Merge LoRA Adapter (Optional)
#@markdown If you trained with LoRA, this cell merges the adapter into a single standalone model file.

%cd /content/chatterbox-finetuning

print("Merging LoRA adapter into base model...")
print("This creates a standalone .safetensors file that doesn't require LoRA adapters.")

!python merge_lora.py

print("\n✅ Merge completed!")
print("Your merged model is ready for production use.")
print("Check chatterbox_output/ for the merged .safetensors file.")